FILTRAGGIO

In [ ]:
import pandas as pd
from datetime import datetime
from pathlib import Path
import hashlib
import pandas as pd
from datetime import datetime
from pathlib import Path
import librosa
import noisereduce as nr
import soundfile as sf

In [2]:
INPUT_AUDIO_DIR = Path("AudioSamples")
OUTPUT_AUDIO_DIR = Path("AudioSamplesPreprocessed")
OUTPUT_AUDIO_DIR.mkdir(parents=True, exist_ok=True)
CSV_OUTPUT_PATH = "audio_samples_metadata.csv"

In [3]:
METADATA_KEYS = ["tmst", "noId", "blvl", "rmsv"]

def getMetadata(wav_bytes):
    metadata = {}
    i = 12  # salta RIFF + size + WAVE

    size = len(wav_bytes)

    while i + 8 <= size:
        chunk_id = wav_bytes[i:i+4]
        chunk_size = int.from_bytes(wav_bytes[i+4:i+8], "little")
        chunk_data_start = i + 8
        chunk_data_end = chunk_data_start + chunk_size

        if chunk_id == b"LIST" and wav_bytes[chunk_data_start:chunk_data_start+4] == b"INFO":
            j = chunk_data_start + 4  # salta "INFO"

            while j + 8 <= chunk_data_end:
                key = wav_bytes[j:j+4].decode("ascii", errors="ignore").strip()
                length = int.from_bytes(wav_bytes[j+4:j+8], "little")
                value_bytes = wav_bytes[j+8:j+8+length]

                # rimuove \x00 finali
                value = value_bytes.rstrip(b"\x00").decode("ascii", errors="ignore")
                if key in METADATA_KEYS:
                    metadata[key] = value
                else:
                    raise Exception(f"Unexpected metadata key: {key}")
                j += 8 + length

        # chunk allineati a 2 byte
        i = chunk_data_end + (chunk_size % 2)

    

    return metadata

In [4]:
# 1. Configurazione Parametri
audio_samples_source_directory = Path(INPUT_AUDIO_DIR)
date_format = "%Y/%m/%d,%H:%M:%S"

# Specifica qui l'intervallo con lo stesso formato dei metadati
start_str = "2026/03/09,00:00:00"
end_str   = "2026/03/13,23:59:59"

# Conversione delle stringhe di input in oggetti datetime per il confronto
inizio = datetime.strptime(start_str, date_format)
fine = datetime.strptime(end_str, date_format)

# 2. Scansione e Filtrazione
audio_samples = []

if audio_samples_source_directory.exists():
    for input_path in audio_samples_source_directory.glob("*.wav"):
        try:
            with open(input_path, "rb") as f:
                audio_content = f.read()
                # Uso della tua funzione di estrazione
                metadata = getMetadata(audio_content)
                timestamp = metadata.get("tmst")
                
                if timestamp:
                    datetime_file = datetime.strptime(timestamp, date_format)
                    
                    # Filtro temporale
                    if inizio <= datetime_file <= fine:
                        sha256_hash = hashlib.sha256(audio_content).hexdigest()
                        audio_samples.append({
                            "original_filename": input_path.name,
                            "filename": f"audio_{sha256_hash}.wav",
                            "timestamp": datetime_file,
                            })
        except Exception as e:
            print(f"Errore nell'elaborazione di {input_path.name}: {e}")
else:
    print(f"Errore: La cartella '{audio_samples_source_directory}' non esiste.")

# 3. Creazione del DataFrame
audiofiles_df = pd.DataFrame(audio_samples, columns=["original_filename", "filename", "timestamp"])

if not audiofiles_df.empty:
    print(f"Trovati {len(audiofiles_df)} file nell'intervallo selezionato.")
    audiofiles_df = audiofiles_df.sort_values(by="timestamp", ascending=True)
    audiofiles_df = audiofiles_df.reset_index(drop=True)
else:
    print("Nessun file trovato nell'intervallo specificato.")

audiofiles_df.head()

Trovati 764 file nell'intervallo selezionato.


,original_filename,filename,timestamp
0,audio_548329df66f2be5cebe3a3ab8688da8658eefb65...,audio_548329df66f2be5cebe3a3ab8688da8658eefb65...,2026-03-09 15:34:22
1,audio_4308ece211efbbe098f438474dffb8efb1517af2...,audio_4308ece211efbbe098f438474dffb8efb1517af2...,2026-03-09 15:39:28
2,audio_be49fb71e4c2a540ba07d72d6d59d423ad4e15bd...,audio_be49fb71e4c2a540ba07d72d6d59d423ad4e15bd...,2026-03-09 15:41:41
3,audio_bf2de73f90b6f7dd200b4a58d1e8ac2f7186900a...,audio_bf2de73f90b6f7dd200b4a58d1e8ac2f7186900a...,2026-03-09 15:46:47
4,audio_5919f6a394c4aa055ae46794683e94673cee6288...,audio_5919f6a394c4aa055ae46794683e94673cee6288...,2026-03-09 15:51:54


In [5]:
# Trova tutte le occorrenze dei timestamp duplicati
# keep=False assicura di vedere TUTTI i file coinvolti nel duplicato, non solo il secondo
duplicati = audiofiles_df[audiofiles_df.duplicated(subset=['timestamp'], keep=False)]

# Ordiniamo per timestamp per vederli vicini
duplicati_ordinati = duplicati.sort_values(by='timestamp')

if not duplicati_ordinati.empty:
    print(f"Attenzione: Trovati {len(duplicati_ordinati)} file con timestamp identici.")
    display(duplicati_ordinati)
else:
    print("Ottimo! Non ci sono file con timestamp duplicati.")
    

Ottimo! Non ci sono file con timestamp duplicati.


In [6]:
audiofiles_df.to_csv(CSV_OUTPUT_PATH, index=False)

RIMOZIONE RUMORE

In [7]:
SAMPLE_RATE = 16000   #16 kHz
FRAME_SIZE = 1024
HOP_LENGTH = 512
BASE_SPLIT_FREQUENCY = 2000

def clean_audio_signal(raw_signal, noise_signal, sample_rate):
    return nr.reduce_noise(
        y=raw_signal, 
        sr=sample_rate, 
        y_noise=noise_signal, 
        stationary=True
) 

In [8]:
noise_signal, noise_sr = librosa.load("noise.wav", sr=SAMPLE_RATE)

for index, row in audiofiles_df.iterrows():

    original_filename = row["original_filename"]
    new_filename = row["filename"]

    # 2. Costruiamo il percorso completo (AudioSamples/nome_file.wav)
    input_path = INPUT_AUDIO_DIR / original_filename
    output_path = OUTPUT_AUDIO_DIR / new_filename

    # 3. Carichiamo l'audio con librosa
    raw_audio_signal, sr = librosa.load(input_path, sr=SAMPLE_RATE)

    print(f"File caricato: {output_path} (timestamp: {row['timestamp']})")
    #print(f"Frequenza di campionamento (SR): {sr} Hz")
    #print(f"Durata: {librosa.get_duration(y=raw_audio_signal, sr=sr):.2f} secondi")
    #print(f"Timestamp del file: {row['timestamp']}")

    cleaned_audio_signal=clean_audio_signal(raw_audio_signal, noise_signal, SAMPLE_RATE)
    sf.write(output_path, cleaned_audio_signal, sr)

print("Preprocessing completato. File salvati in:", OUTPUT_AUDIO_DIR)


File caricato: AudioSamplesPreprocessed\audio_548329df66f2be5cebe3a3ab8688da8658eefb653bb0d37c1c52dc30049b7f2f.wav (timestamp: 2026-03-09 15:34:22)
File caricato: AudioSamplesPreprocessed\audio_4308ece211efbbe098f438474dffb8efb1517af2e8fa7c8249f6a4b2a00082be.wav (timestamp: 2026-03-09 15:39:28)
File caricato: AudioSamplesPreprocessed\audio_be49fb71e4c2a540ba07d72d6d59d423ad4e15bd7177573158ed508d1755bf89.wav (timestamp: 2026-03-09 15:41:41)
File caricato: AudioSamplesPreprocessed\audio_bf2de73f90b6f7dd200b4a58d1e8ac2f7186900a35f8d3008699f6c5aa9745f3.wav (timestamp: 2026-03-09 15:46:47)
File caricato: AudioSamplesPreprocessed\audio_5919f6a394c4aa055ae46794683e94673cee6288eee3609bbb985f1b286d92ca.wav (timestamp: 2026-03-09 15:51:54)
File caricato: AudioSamplesPreprocessed\audio_fffdb5d3721332d48f2a73d0ff531aa16961ea0c71e712320b8423789b2c07ed.wav (timestamp: 2026-03-09 15:57:01)
File caricato: AudioSamplesPreprocessed\audio_4b18ffc1dbec674e4f60467e43bf3139ef8f31f7ab2df256be90061358b5d756.w